# IC-0921 — How diversified is your client, really?

**ML & FinTech · 115-1 · 20260921 · 50 minutes: 40 working, 10 discussion**

| Step | Min | What you do |
|---|---|---|
| 1 | 16 | Get to know the data, and learn the two clustering tools |
| 2 | 14 | Cluster the client's portfolio, two ways |
| 3 | 10 | Give the client advice |
| — | 10 | Discussion |

You work at a robo-advisor. A client calls:

> *"I'm well diversified. I own twelve different stocks."*

Twelve tickers is not twelve bets. If several of them rise and fall together, the client owns
fewer independent positions than he thinks. **By the end you will tell him how many bets he
actually owns.**

You may use any AI tool. It will write the code. The code is not what is graded — whether you
understand what the output means is.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

---
# Step 1 — Know your data, learn the tools (16 min)

## 1a. What the client owns

Run the cell. The table gives every ticker, the company behind it, and its industry.
**Keep this table in view for the rest of the exercise.**

In [ ]:
CAND = ["historical_stock_prices_2017.csv",
        "../slides/historical_stock_prices_2017.csv",
        "../00-course-info/in-class-exercise/historical_stock_prices_2017.csv"]
path = next((p for p in CAND if Path(p).exists()), None)
if path is None:
    raise FileNotFoundError("Put historical_stock_prices_2017.csv next to this notebook.")

prices = pd.read_csv(path, index_col=0)
prices.index = pd.to_datetime(prices.index)

holdings = pd.DataFrame([
    ("BANC", "Banc of California",        "Banking"),
    ("BANF", "BancFirst Corporation",     "Banking"),
    ("GWB",  "Great Western Bancorp",     "Banking"),
    ("ISTR", "Investar Holding Corp.",    "Banking"),
    ("CAKE", "The Cheesecake Factory",    "Restaurants"),
    ("DFRG", "Del Frisco's Restaurant Gp.","Restaurants"),
    ("LOCO", "El Pollo Loco Holdings",    "Restaurants"),
    ("BREW", "Craft Brew Alliance",       "Brewing"),
    ("AAPL", "Apple Inc.",                "Technology"),
    ("EBAY", "eBay Inc.",                 "E-commerce"),
    ("QQQ",  "Invesco QQQ Trust",         "ETF, Nasdaq-100"),
    ("CCOI", "Cogent Communications",     "Telecom"),
], columns=["ticker", "company", "industry"]).set_index("ticker")

PORTFOLIO = holdings.index.tolist()
P = prices[PORTFOLIO]          # 251 trading days of 2017 x 12 stocks
print(P.shape)
holdings

## 1b. Prices through the year

In [ ]:
P.plot(figsize=(11, 5), linewidth=1)
plt.title("Daily closing prices, 2017")
plt.ylabel("price (USD)")
plt.legend(ncol=4, fontsize=8)
plt.show()

## 1c. Daily returns

Prices tell you the level. **Returns tell you the movement**, which is what risk is made of.

In [ ]:
R = np.log(P).diff().dropna()     # daily log returns

R.plot(subplots=True, layout=(4, 3), figsize=(12, 8),
       sharey=True, legend=False, linewidth=0.7, title=list(R.columns))
plt.tight_layout()
plt.show()

## 1d. Summary statistics

In [ ]:
summary = pd.DataFrame({
    "mean price":    P.mean(),
    "min price":     P.min(),
    "max price":     P.max(),
    "ann. return %": R.mean() * 252 * 100,
    "ann. vol %":    R.std() * np.sqrt(252) * 100,
}).round(1)
summary.join(holdings[["industry"]])

## 1e. Risk against reward

Each stock as one point: risk on the horizontal axis, reward on the vertical.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(summary["ann. vol %"], summary["ann. return %"], s=60)
for t, row in summary.iterrows():
    plt.annotate(t, (row["ann. vol %"], row["ann. return %"]),
                 xytext=(4, 4), textcoords="offset points")
plt.axhline(0, color="grey", linewidth=0.8)
plt.xlabel("annualised standard deviation (%)   — risk")
plt.ylabel("annualised return (%)   — reward")
plt.title("Risk and return, 2017")
plt.show()

### Questions on the data

**Q1.** Which stock has the **lowest** annualised volatility of all twelve? Look up what it is in
the holdings table. Why would that kind of holding be less volatile than a single company?

> _replace this line_

**Q2.** Four of the twelve are banks. From the returns plots and the risk–return chart, would you
expect all four to move together? Write down your guess now — you will test it in Step 2.

> _replace this line_

## 1f. The two tools, on something small

Before the portfolio, practise on eight credit card customers where you can see the answer with
your own eyes. The code is complete — run it and read the output.

In [ ]:
cust = pd.DataFrame({
    "customer": ["Amy", "Ben", "Cora", "Dan", "Eve", "Fay", "Gus", "Hal"],
    "spend":    [5, 6, 4, 5, 28, 30, 32, 29],   # monthly card spending, NT$ thousand
    "trans":    [4, 5, 3, 6, 22, 25, 20, 24],   # card transactions per month
}).set_index("customer")

cust["kmeans"] = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(cust)

plt.scatter(cust["spend"], cust["trans"], c=cust["kmeans"], s=100)
for name, row in cust.iterrows():
    plt.annotate(name, (row["spend"], row["trans"]))
plt.xlabel("monthly spending (NT$ thousand)"); plt.ylabel("transactions per month")
plt.show()
cust

Now the same eight customers as a **dendrogram**. Read it from the bottom up: the height at which two branches join tells you how different they are.

In [ ]:
Zc = linkage(cust[["spend", "trans"]].values, method="average")
dendrogram(Zc, labels=cust.index.tolist())
plt.ylabel("height at which groups join")
plt.show()

cust["tree"] = fcluster(Zc, 2, criterion="maxclust")
cust

**Q3.** In the dendrogram, the last two groups join very high up while everything before joined
low down. What does that big jump tell you about how many groups these customers really form?

> _replace this line_

**Q4.** Change `n_clusters=2` to `3` and re-run 1f. Who moves? Looking at the plot, is that a real
group or is k-means forcing one?

> _replace this line_

In [ ]:
# Q4 — try K = 3 here



---
# Step 2 — Cluster the portfolio (14 min)

You are clustering **stocks**, not days. Each stock must be one row, as each customer was one row.
`P` has stocks in its columns, so use `P.T` whenever you cluster stocks.

## 2a. The obvious approach (6 min)

Run **k-means with K = 3** on the price series exactly as they are. Print the members of each cluster.

In [ ]:
# 2a



**Q5.** Look at the industries of the stocks inside each cluster, using the holdings table.
What do the members of a cluster actually have in common? If it is nothing about the businesses,
say what it is instead. *Hint: compare with the `mean price` column of your summary table.*

> _replace this line_

**Q6.** Would you show these three groups to the client as his three bets? Yes / No, and why.

> _replace this line_

## 2b. Cluster on what actually matters (8 min)

Two stocks belong in the same bet when they **move together**, not when they cost the same.

```python
C = R.corr()                       # how much each pair moves together
D = 1 - C                          # turn correlation into a distance
Z = linkage(squareform(D.values, checks=False), method="average")
dendrogram(Z, labels=D.columns.tolist())
```

Draw the dendrogram, then cut it into 3 groups with `fcluster`, as in Step 1.

In [ ]:
# 2b



**Q7.** Write down your three groups.

> _____

**Q8.** Go back to your guess in Q2. Did the four banks end up together? If one of them did not,
name it — and note that this is a fact about the data, not a mistake.

> _____

---
# Step 3 — Advise the client (10 min)

**Q9.** How many independent bets does the client actually own?

> _____

**Q10.** Look at AAPL and QQQ in your dendrogram, and at what QQQ is in the holdings table.
Should the client hold both? Why?

> _____

**Q11.** He wants to cut from twelve holdings to four. **Name the four you would keep**, with one
sentence of reasoning from your own dendrogram. Four tickers — he is on the phone.

> Keep: _____ , _____ , _____ , _____
>
> Because: _____

---
## Before the discussion

Restart & Run All, check every blank is filled, push as `IC-0921.ipynb`.

Be ready to answer out loud: **the client owns twelve stocks. Is he diversified?**